In [ ]:
#just change date in filepath to load different year
from pathlib import Path
import pandas as pd

BASE_DIR = #path to folder containing data
OUTPUT_FILE = #path to where you want merged file to go

AIRPORTS = [
    "TPA","STL","SNA","SMF","SLC","SJC","SFO","SEA","SAT","SAN","RSW","RDU","PIT",
    "PHX","PHL","PDX","ORD","OGG","OAK","MSY","MSP","MIA","MDW","MCO","MCI","LGA",
    "LAX","LAS","JFK","IND","IAH","IAD","HOU","HNL","FLL","EWR","DTW","DFW","DEN",
    "DCA","DAL","CVG","CMH","CLT","CLE","BWI","BOS","BNA","AUS","ATL"
]

dfs = []
#downloading data gives folders based on airport, so go through and get 
#data from each folder
for airport in AIRPORTS:
    airport_path = BASE_DIR / airport
    file_path = airport_path / f"{airport}_2025_eGSE_8760_load.csv"

    #noify if file doesn't exist for an airport
    if not file_path.exists():
        print(f"Missing file for {airport}, skipping...")
        continue

    print(f"Processing {airport}...")

    df = pd.read_csv(file_path)
    cols = df.columns.tolist()

    # format for different column titles 
    #(since it changed in raw data)
    if "Hour of Day" in cols:
        hour_series = df["Hour of Day"]
        date_series = df["Date"]
        airport_series = df["Airport Code"]
        egse_series = df["eGSE %"]
        demand_series = df["Demand (kW)"]

    elif "Hour" in cols:
        hour_series = df["Hour"]
        date_series = df["Date"]
        airport_series = df["Airport"]
        egse_series = df["eGSE_share"]

        # convert MW → kW
        demand_series = df["Demand_MW"] * 1000

    else:
        raise ValueError(f"Unknown format in {file_path}: {cols}")
    
    # format date (dd/mm/25)
    date_series = pd.to_datetime(date_series, errors="coerce")
    date_series = date_series.apply(lambda d: d.replace(year=2025) if pd.notnull(d) else d)
    date_series = date_series.dt.strftime("%d/%m/%y")


    #format hour
    hour_formatted = (
        hour_series.astype(int)
        .astype(str)
        .str.zfill(2) + ":00"
    )

    # create final output structure
    df_final = pd.DataFrame({
        "FlightDate": date_series,
        "Hour": hour_formatted,
        "Origin": airport_series,
        "eGSE_percent": egse_series,
        "Demand_kW": demand_series
    })

    dfs.append(df_final)

#raise error if no data
if not dfs:
    raise ValueError("No data was loaded. Check file paths.")

#combine all airport data 
final_df = pd.concat(dfs, ignore_index=True)

# sort like flight dataset
final_df = final_df.sort_values(["Origin", "FlightDate", "Hour"])

#write output
final_df.to_csv(OUTPUT_FILE, index=False)

print(f"\nSaved to {OUTPUT_FILE}")
print(f"Total rows: {len(final_df)}")

Processing TPA...
Processing STL...
Processing SNA...
Processing SMF...
Processing SLC...
Processing SJC...
Processing SFO...
Processing SEA...
Processing SAT...
Processing SAN...
Processing RSW...
Processing RDU...
Processing PIT...
Processing PHX...
Processing PHL...
Processing PDX...
Processing ORD...
Processing OGG...
Processing OAK...
Processing MSY...
Processing MSP...
Processing MIA...
Processing MDW...
Processing MCO...
Processing MCI...
Processing LGA...
Processing LAX...
Processing LAS...
Processing JFK...
Processing IND...
Processing IAH...
Processing IAD...
Processing HOU...
Processing HNL...
Processing FLL...
Processing EWR...
Processing DTW...
Processing DFW...
Processing DEN...
Processing DCA...
Processing DAL...
Processing CVG...
Processing CMH...
Processing CLT...
Processing CLE...
Processing BWI...
Processing BOS...
Processing BNA...
Processing AUS...
Processing ATL...

Saved to C:\Users\alecg\Downloads\egse_data\2025_combined_egse_load_data.csv
Total rows: 2190000
